In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv("new_train.csv")

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         75000 non-null  float64
 1   Value         74060 non-null  float64
 2   Unit          74059 non-null  object 
 3   cleaned_text  74990 non-null  object 
dtypes: float64(2), object(2)
memory usage: 2.3+ MB


In [7]:
df=df.drop(["combined_text","important_word"],axis=1)

In [9]:
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoConfig, AutoModelForSequenceClassification
from datasets import Dataset
import numpy as np
import torch
import pandas as pd


In [11]:
train_df = pd.read_csv("new_train.csv")  # or your existing df

# Fill missing values safely
train_df["cleaned_text"] = train_df["cleaned_text"].fillna("")
train_df["Value"] = train_df["Value"].fillna("")
train_df["Unit"] = train_df["Unit"].fillna("")

# ✅ Combine all features into text input
# Example: "text [SEP] Value: 200 Unit: ml"
train_df["text"] = (
    train_df["cleaned_text"].astype(str)
    + " [SEP] Value: " + train_df["Value"].astype(str)
    + " Unit: " + train_df["Unit"].astype(str)
)

# Keep only needed columns
train_df = train_df[["text", "price"]].dropna().reset_index(drop=True)


In [12]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# -----------------------------
# HuggingFace Dataset
# -----------------------------
hf_ds = Dataset.from_pandas(train_df)
hf_ds = hf_ds.map(preprocess, batched=True)
hf_ds = hf_ds.rename_column("price", "labels")
hf_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# -----------------------------
# Split for validation
# -----------------------------
split = hf_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
val_ds = split["test"]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/75000 [00:00<?, ? examples/s]

In [19]:
# The model trained for 2 epochs lives inside "bert-price" (from previous run)
config = AutoConfig.from_pretrained(model_name, num_labels=1)
model = AutoModelForSequenceClassification.from_pretrained("bert-price/checkpoint-16876", config=config)


In [20]:
training_args = TrainingArguments(
    output_dir="bert-price",       # same folder = continues training
    num_train_epochs=2,            # total 4 epochs, will continue from checkpoint
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    logging_dir="logs",
    logging_steps=100,
    save_total_limit=2,
)

In [ ]:
# -----------------------------
# SMAPE metric
# -----------------------------
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.flatten()
    labels = labels.flatten()
    smape = 100 * np.mean(
        2 * np.abs(preds - labels) / (np.abs(preds) + np.abs(labels) + 1e-8)
    )
    return {"smape": smape}

# -----------------------------
# Trainer (continue training)
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

# -----------------------------
# Continue training
# -----------------------------
trainer.train()

# -----------------------------
# Evaluate after completion
# -----------------------------
metrics = trainer.evaluate()
print("✅ Final Metrics:", metrics)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shashankyadavriiii (shashankyadavriiii-indian-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU mode")


True
Tesla T4


In [2]:
!pip install -q transformers datasets torch accelerate

In [3]:
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoConfig, AutoModelForSequenceClassification
from datasets import Dataset
import numpy as np
import pandas as pd
import torch

# -----------------------------
# 1️⃣ Load & combine data
# -----------------------------
train_df = pd.read_csv("new_train.csv")

train_df["cleaned_text"] = train_df["cleaned_text"].fillna("")
train_df["Value"] = train_df["Value"].fillna("")
train_df["Unit"] = train_df["Unit"].fillna("")

train_df["text"] = (
    train_df["cleaned_text"].astype(str)
    + " [SEP] Value: " + train_df["Value"].astype(str)
    + " Unit: " + train_df["Unit"].astype(str)
)

train_df = train_df[["text", "price"]].dropna().reset_index(drop=True)

# -----------------------------
# 2️⃣ Tokenizer
# -----------------------------
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

hf_ds = Dataset.from_pandas(train_df)
hf_ds = hf_ds.map(preprocess, batched=True)
hf_ds = hf_ds.rename_column("price", "labels")
hf_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# -----------------------------
# 3️⃣ Split data
# -----------------------------
split = hf_ds.train_test_split(test_size=0.1, seed=42)
train_ds, val_ds = split["train"], split["test"]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/75000 [00:00<?, ? examples/s]

In [7]:
config = AutoConfig.from_pretrained(model_name, num_labels=1)
model = AutoModelForSequenceClassification.from_pretrained("bert-price/checkpoint-8438", config=config)

# -----------------------------
# 5️⃣ Optimized training args (FAST)
# -----------------------------
training_args = TrainingArguments(
    output_dir="bert-price",
    num_train_epochs=2,               # only 2 more epochs
    per_device_train_batch_size=16,   # good for T4, adjust to 8 if OOM
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    fp16=True,                        # mixed precision → faster
    dataloader_pin_memory=False,      # avoid CPU warning
    logging_dir="logs",
    logging_steps=100,
    report_to=["wandb"],              # ✅ log to W&B
    save_total_limit=2,
    gradient_accumulation_steps=1,    # can increase to 2 if memory tight
)

# -----------------------------
# 6️⃣ Metric (SMAPE)
# -----------------------------
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.flatten()
    labels = labels.flatten()
    smape = 100 * np.mean(
        2 * np.abs(preds - labels) / (np.abs(preds) + np.abs(labels) + 1e-8)
    )
    return {"smape": smape}

# -----------------------------
# 7️⃣ Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

# -----------------------------
# 8️⃣ Resume training on GPU
# -----------------------------
print("✅ Device:", "GPU" if torch.cuda.is_available() else "CPU")
trainer.train()

# -----------------------------
# 9️⃣ Evaluate
# -----------------------------
metrics = trainer.evaluate()
print("🏁 Final metrics:", metrics)

✅ Device: GPU


Epoch,Training Loss,Validation Loss,Smape
1,295.061000,667.665710,53.937511
2,288.424400,632.424133,52.534294


🏁 Final metrics: {'eval_loss': 632.4241333007812, 'eval_smape': 52.53429412841797, 'eval_runtime': 8.7673, 'eval_samples_per_second': 855.45, 'eval_steps_per_second': 106.988, 'epoch': 2.0}


In [8]:
df_test=pd.read_csv("test2.csv")
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      75000 non-null  int64  
 1   sample_id       75000 non-null  int64  
 2   image_link      75000 non-null  object 
 3   Item Name       74992 non-null  object 
 4   Value           75000 non-null  float64
 5   Unit            75000 non-null  object 
 6   Bullet Point 1  64916 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 4.0+ MB


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
import pandas as pd
import numpy as np

# --- 1️⃣ Load your test data ---
df_test = pd.read_csv("test2.csv")

# Combine text the same way as during training
df_test['text'] = (
    df_test['Item Name'].fillna('') + ' ' + df_test['Bullet Point 1'].fillna('')
)

# --- 2️⃣ Load tokenizer and model ---
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained("bert-price/checkpoint-8438")  # your trained model directory
model.eval()

# --- 3️⃣ Preprocess test data ---
def preprocess(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

hf_test = Dataset.from_pandas(df_test)
hf_test = hf_test.map(preprocess, batched=True)
hf_test.set_format(type='torch', columns=['input_ids', 'attention_mask'])

# --- 4️⃣ Run inference ---
preds = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for batch in torch.utils.data.DataLoader(hf_test, batch_size=16):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        batch_preds = outputs.logits.squeeze().cpu().numpy()
    preds.extend(batch_preds)

# --- 5️⃣ Store predictions ---
df_test["predicted_price"] = preds

# --- 6️⃣ Optionally add SMAPE metric if you have true Value ---
if "Value" in df_test.columns:
    y_true = df_test["Value"].values
    y_pred = df_test["predicted_price"].values
    smape = 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_pred) + np.abs(y_true) + 1e-8))
    print(f"SMAPE on test set: {smape:.2f}%")

# --- 7️⃣ Save results ---
df_test[["sample_id", "predicted_price"]].to_csv("predictions.csv", index=False)
print("✅ Predictions saved to predictions.csv")


Map:   0%|          | 0/75000 [00:00<?, ? examples/s]

SMAPE on test set: 99.13%
✅ Predictions saved to predictions.csv


In [15]:
df3=pd.read_csv("predictions.csv")

In [16]:
df3.rename(columns={"predicted_price": "price"}, inplace=True)


In [17]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  75000 non-null  int64  
 1   price      75000 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 1.1 MB


In [18]:
df4=pd.read_csv("submission_bert_new.csv")

In [19]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  75000 non-null  int64  
 1   price      75000 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 1.1 MB


In [20]:
df3.to_csv("final_pred.csv")